# HW3 Advanced Quantum Probability

PDF pages 49-66: tensor/Kronecker product, reciprocity, LTP/interference, order effects, density operators, sequential density measurements, and POVM checks.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import csv
import numpy as np
np.set_printoptions(precision=4, suppress=True)

from quantum_lib import (
    conditional_state, density_conditional_state, density_probability,
    density_sequence_probability, event_probability, interference, is_povm, kron,
    mixed_density, order_effects, projector, pure_density, ray_projector,
    sequential_probability, two_dim_unitary,
)

## Tensor Product


In [2]:
A = np.array([[1, 2], [3, 4]], dtype=complex)
B = np.array([[0, 5], [6, 7]], dtype=complex)
print('A tensor B=\n', kron(A, B))

A tensor B=
 [[ 0.+0.j  5.+0.j  0.+0.j 10.+0.j]
 [ 6.+0.j  7.+0.j 12.+0.j 14.+0.j]
 [ 0.+0.j 15.+0.j  0.+0.j 20.+0.j]
 [18.+0.j 21.+0.j 24.+0.j 28.+0.j]]


## Law of Reciprocity for One-Dimensional Events


In [3]:
a = np.array([1, 0], dtype=complex)
b = two_dim_unitary(np.deg2rad(50), 0)[:, 0]
PA = ray_projector(a)
PB = ray_projector(b)
q_A_given_B = sequential_probability(b, PA) / event_probability(b, PB)
q_B_given_A = sequential_probability(a, PB) / event_probability(a, PA)
print('q(A|B)=', q_A_given_B)
print('q(B|A)=', q_B_given_A)
print('reciprocity holds:', np.allclose(q_A_given_B, q_B_given_A))

q(A|B)= 0.8213938048432696
q(B|A)= 0.8213938048432695
reciprocity holds: True


## LTP, Interference, and Order Effects


In [4]:
psi = np.array([np.sqrt(.55), np.sqrt(.45)], dtype=complex)
PA = projector([0], 2)
PnotA = projector([1], 2)
U = two_dim_unitary(np.deg2rad(65), np.deg2rad(20))
PD = ray_projector(U[:, 0])
PnotD = np.eye(2) - PD

print('D direct, D total, Int(D):', interference(psi, [PA, PnotA], PD))
print('not-D direct, total, Int(not-D):', interference(psi, [PA, PnotA], PnotD))
print('Int(D)+Int(not-D)=', interference(psi, [PA, PnotA], PD)[2] + interference(psi, [PA, PnotA], PnotD)[2])
print('O1 q(A,D)-q(D,A):', order_effects(psi, PA, PD))
print('O2 q(notA,D)-q(D,notA):', order_effects(psi, PnotA, PD))

D direct, D total, Int(D): (0.9448218064587911, 0.5211309130870351, 0.423690893371756)
not-D direct, total, Int(not-D): (0.05517819354120918, 0.47886908691296504, -0.42369089337175586)
Int(D)+Int(not-D)= 1.6653345369377348e-16
O1 q(A,D)-q(D,A): (0.3912200219786924, 0.6720603779795564, -0.28084035600086404)
O2 q(notA,D)-q(D,notA): (0.1299108911083427, 0.2727614284792346, -0.14285053737089193)


## Density Operators and Sequential Measurements


In [5]:
rho_pure = pure_density(psi)
rho_mixed = mixed_density([np.array([1, 0]), np.array([0, 1])], probabilities=[.7, .3])

print('trace pure rho:', np.trace(rho_pure))
print('trace mixed rho:', np.trace(rho_mixed))
print('q(D) from density:', density_probability(rho_pure, PD))
print('q(A,D) from density:', density_sequence_probability(rho_pure, PA, PD))
print('rho | A =\n', density_conditional_state(rho_pure, PA))

trace pure rho: (1+0j)
trace mixed rho: (1+0j)
q(D) from density: 0.944821806458791
q(A,D) from density: 0.39122002197869243
rho | A =
 [[1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j]]


## POVM Example


In [6]:
# Five noisy rating measurement operators. Column j encodes response noise around true state j.
n = 5
sigma = .65
x = np.arange(n)
weights = np.vstack([np.exp(-0.5 * ((x - j) / sigma) ** 2) for j in range(n)])
weights = weights / weights.sum(axis=0, keepdims=True)
operators = [np.diag(np.sqrt(weights[i])) for i in range(n)]

print('is POVM:', is_povm(operators))
state_R2 = np.eye(n)[:, 1]
print('response probabilities for true R2:', [event_probability(state_R2, M) for M in operators])

is POVM: True
response probabilities for true R2: [0.18888039510100937, 0.6168006877313569, 0.18888039510100937, 0.005423916340643947, 1.460572598038567e-05]
